# Stage 2 — Combined Analysis

This is the centrepiece analysis notebook. It brings together safety and faithfulness results to make the causal argument.

## The causal argument

Stage 1 showed that certain experts are *correlated* with safe/faithful behaviour (high |RD|). Correlation alone is not causation — those experts could be passengers rather than drivers.

Stage 2 tests causation by suppressing those experts and observing whether behaviour changes predictably:

| Intervention | Expected if causal | Null expectation |
|---|---|---|
| Suppress compliance experts | safe_rate increases | no change |
| Suppress refusal experts | safe_rate decreases | no change |
| Suppress confabulation experts | faithfulness increases | no change |

If both directions of safety steering work (safe_rate increases *and* decreases depending on which experts are suppressed), this is strong bidirectional causal evidence.

In [ ]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

sns.set_theme(style='whitegrid', font='serif')
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 12,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'figure.dpi': 150,
})

RESULTS_PATH = '/scratch/sc23jc3/stage2_results/results.json'
OUT_DIR = 'figures'
os.makedirs(OUT_DIR, exist_ok=True)

with open(RESULTS_PATH) as f:
    results = json.load(f)

safety = results['safety']
faith  = results['faithfulness']


def get_rate(d, condition):
    """Extract rate from either {records, safe_rate} dict or plain float."""
    val = d.get(condition, float('nan'))
    if isinstance(val, dict):
        return val.get('safe_rate', float('nan'))
    return val


def get_delta(d, cond='hard'):
    b = get_rate(d, 'baseline')
    v = get_rate(d, cond)
    if np.isnan(b) or np.isnan(v):
        return float('nan')
    return v - b

## 1. Effect Size Overview — All Conditions

A single plot showing the delta (hard − baseline) for every condition. Positive = effect in the expected direction.

In [ ]:
conditions = [
    ('Safety: suppress\ncompliance experts',  safety['safe_steering']),
    ('Safety: suppress\nrefusal experts',     safety['unsafe_steering']),
    ('Faithfulness:\nsuppress confabulation', faith.get('counterfactual', {})),
    ('Faithfulness:\nunanswerable',           faith.get('unanswerable', {})),
    ('Control:\nSQuAD',                       faith.get('mctest', {})),
]

labels      = [c[0] for c in conditions]
hard_deltas = [get_delta(c[1], 'hard') for c in conditions]
soft_deltas = [get_delta(c[1], 'soft') for c in conditions]

x = np.arange(len(labels))
w = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - w/2, hard_deltas, width=w, label='Hard steering', color='#2980b9', alpha=0.85)
ax.bar(x + w/2, soft_deltas, width=w, label='Soft steering', color='#8e44ad', alpha=0.70, hatch='//')
ax.axhline(0, color='black', linewidth=1.0)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel('Delta vs Baseline')
ax.set_title('Effect of Expert Steering Across All Conditions (Hard vs Soft)')
ax.legend()

for i, (dh, ds) in enumerate(zip(hard_deltas, soft_deltas)):
    for d, xpos in [(dh, x[i] - w/2), (ds, x[i] + w/2)]:
        if not np.isnan(d):
            sign = '+' if d >= 0 else ''
            ax.text(xpos, d + (0.005 if d >= 0 else -0.015), f'{sign}{d:.3f}',
                    ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'combined_effect_sizes.png'), dpi=300, bbox_inches='tight')
plt.show()

## 2. Asymmetry Analysis — Safety

Does suppressing compliance experts raise safe_rate by the same magnitude that suppressing refusal experts lowers it? Symmetry would suggest the experts cleanly partition safe vs unsafe behaviour.

In [ ]:
for cond in ['hard', 'soft']:
    delta_safe   = get_delta(safety['safe_steering'],   cond)
    delta_unsafe = get_delta(safety['unsafe_steering'], cond)

    print(f'\n--- {cond.upper()} MODE ---')
    print(f'Safe steering delta:   {delta_safe:+.4f}  (suppress compliance experts → expect positive)')
    print(f'Unsafe steering delta: {delta_unsafe:+.4f}  (suppress refusal experts → expect negative)')

    if not (np.isnan(delta_safe) or np.isnan(delta_unsafe)):
        asymmetry = abs(delta_safe) - abs(delta_unsafe)
        print(f'Asymmetry (|safe| − |unsafe|): {asymmetry:+.4f}')
        if abs(asymmetry) < 0.05:
            print('  → Near-symmetric.')
        elif asymmetry > 0:
            print('  → Compliance suppression has larger effect.')
        else:
            print('  → Refusal suppression has larger effect.')

## 3. Control Check

The SQuAD control score should remain flat. A large drop would suggest the steering degrades general capability rather than specifically targeting the intended behaviour.

In [ ]:
if 'mctest' in faith:
    b = get_rate(faith['mctest'], 'baseline')
    for cond in ['hard', 'soft']:
        v = get_rate(faith['mctest'], cond)
        d = v - b
        print(f'SQuAD control [{cond}] — Baseline: {b:.3f}, {cond.capitalize()}: {v:.3f}, Delta: {d:+.3f}')
        if np.isnan(d):
            print(f'  → {cond} results not yet available.')
        elif abs(d) < 0.05:
            print('  → Control is stable. Steering is specific to the targeted behaviour.')
        elif d < -0.05:
            print('  → Control drops — steering may be degrading general capability.')
        else:
            print('  → Control improves slightly (investigate).')

## 4. Summary Table

In [ ]:
rows = [
    ('Safety: suppress compliance experts', safety['safe_steering'],         'positive'),
    ('Safety: suppress refusal experts',    safety['unsafe_steering'],       'negative'),
    ('Faithfulness: counterfactual',        faith.get('counterfactual', {}), 'positive'),
    ('Faithfulness: unanswerable',          faith.get('unanswerable', {}),   'positive'),
    ('Control: SQuAD',                      faith.get('mctest', {}),         'flat'),
]

print(f"{'Condition':40s} {'Baseline':>10} {'Hard':>10} {'Δ Hard':>10} {'Soft':>10} {'Δ Soft':>10} {'Expected':>10}")
print('-' * 105)
for label, row, expected in rows:
    b  = get_rate(row, 'baseline')
    h  = get_rate(row, 'hard')
    s  = get_rate(row, 'soft')
    dh = h - b if not (np.isnan(b) or np.isnan(h)) else float('nan')
    ds = s - b if not (np.isnan(b) or np.isnan(s)) else float('nan')
    print(f"{label:40s} {b:10.3f} {h:10.3f} {dh:+10.3f} {s:10.3f} {ds:+10.3f} {expected:>10}")

## Discussion

*(Fill in after examining the results.)*

**Overall assessment**:
- Is there consistent evidence of causal expert involvement in both safety and faithfulness?
- Are the effects bidirectional for safety (both directions of steering work)?
- Is the control condition stable, validating that the effect is specific?

**Limitations**:
- n=100 prompts per condition — what would a full-scale run add?
- Hard deactivation is crude; soft suppression may show different dynamics
- Only CANDIDATE_N=10 experts per layer tested; is this the optimal N?

**Next steps**:
- Expand to full dataset sizes
- Introduce soft suppression and compare effect profiles
- Sweep CANDIDATE_N to find the optimal intersection size